# Three factorizations and a curve fit

To solve Ax = b, factor A once, then solve with the factors. Which
factorization depends on what A is. LU works for any nonsingular matrix,
Cholesky only for a symmetric positive definite one, and QR also solves
least-squares problems, where A is tall. This notebook solves one system all
three ways, shows Cholesky refusing a matrix it can't factor, and then fits a
curve, where the choice between QR and the normal equations decides how many
digits survive.

**Uses:** `mtl5.lu`, `mtl5.cholesky`, `mtl5.qr`, `mtl5.ldlt`,
`mtl5.bunch_kaufman`, `mtl5.inv`, `mtl5.inertia`, `mtl5.is_spd`,
`mtl5.condition_number`, `mtl5.convert`.

**From C++:** [`phase05_dense_decompositions/solve_three_ways.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase05_dense_decompositions/solve_three_ways.cpp), [`phase05_dense_decompositions/least_squares_qr.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase05_dense_decompositions/least_squares_qr.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mtl5

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

## One system, three factorizations

The tutorial's matrix has entries 1 / (1 + |i − j|), plus 1 on the diagonal.
It is symmetric positive definite, so all three factorizations apply. The
right-hand side is b = A x with x = (1, 2, 3, 4), so the error of each
solution is known.

Each factorization returns an object that keeps the factors and solves with
them. Factor once, and each further right-hand side costs only the solve.

In [ ]:
n = 4
i, j = np.indices((n, n))
A = 1.0 / (1.0 + np.abs(i - j)) + np.eye(n)
x_true = np.array([1.0, 2.0, 3.0, 4.0])
b = A @ x_true

assert mtl5.is_spd(mtl5.matrix(A))

solutions = {
    "LU": mtl5.lu(A).solve(mtl5.vector(b)).to_numpy(),
    "Cholesky": mtl5.cholesky(A).solve(mtl5.vector(b)).to_numpy(),
    "QR": mtl5.qr(A).solve(mtl5.vector(b)).to_numpy(),
    "inv(A) @ b": mtl5.inv(A).to_numpy() @ b,
}
eps = np.finfo(float).eps
kappa = mtl5.condition_number(mtl5.matrix(A))
print(f"cond(A) = {kappa:.2f}\n")
for name, x in solutions.items():
    error = np.linalg.norm(x - x_true) / np.linalg.norm(x_true)
    print(f"{name:>11}: x = {np.round(x, 12)}, relative error {error:.1e}")
    # a backward-stable solve is accurate to about cond(A) * eps
    assert error <= 10 * kappa * eps, name

All four answers are right to the last digit or two. The case against
`inv(A) @ b` is cost more than accuracy. Forming A⁻¹ takes about 2n³
operations, LU about (2/3)n³, and Cholesky half that. And a sparse matrix has a
dense inverse, so an inverse can't even be stored for the matrices of the
later examples.

## A matrix Cholesky can't factor

Lower the first two diagonal entries to 0.1. The matrix stays symmetric, but
it is no longer positive definite. Cholesky needs a square root of each pivot,
and stops with an error when a pivot isn't positive. LU and QR need neither
symmetry nor definiteness, and still solve.

In [ ]:
A_indef = A.copy()
A_indef[0, 0] = A_indef[1, 1] = 0.1
b_indef = A_indef @ x_true
M = mtl5.matrix(A_indef)
print("symmetric:", mtl5.is_symmetric(M), " SPD:", mtl5.is_spd(M))

try:
    mtl5.cholesky(A_indef)
except RuntimeError as err:
    print("Cholesky:", err)
else:
    raise AssertionError("Cholesky should reject an indefinite matrix")

for name, factor in [("LU", mtl5.lu), ("QR", mtl5.qr)]:
    x = factor(A_indef).solve(mtl5.vector(b_indef)).to_numpy()
    print(f"{name}: error {np.linalg.norm(x - x_true):.1e}")
    assert np.allclose(x, x_true, rtol=0, atol=1e-13)

For a symmetric indefinite matrix there is a factorization in between:
LDLᵀ takes no square roots, so negative pivots are allowed. The signs of D
are the matrix's inertia, its count of positive and negative eigenvalues
(Sylvester's law of inertia). So the factorization also tells how far from
definite the matrix is. One eigenvalue is negative here.

In [ ]:
F = mtl5.ldlt(A_indef)
D = np.asarray(F.diagonal())
x = F.solve(mtl5.vector(b_indef)).to_numpy()
counts = mtl5.inertia(M)
print("D =", np.round(D, 4), "  inertia:", counts)

assert np.allclose(x, x_true, rtol=0, atol=1e-13)
assert (D < 0).sum() == counts["negative"] == 1
assert (D > 0).sum() == counts["positive"] == 3

`ldlt` doesn't pivot, so it fails on a zero pivot even when the matrix is
nonsingular. The smallest example is [[0, 1], [1, 0]]. Bunch–Kaufman is LDLᵀ
with symmetric pivoting, using 2 × 2 blocks in D where a 1 × 1 pivot would be
zero or too small, and it solves that matrix.

In [ ]:
swap = np.array([[0.0, 1.0], [1.0, 0.0]])
try:
    mtl5.ldlt(swap)
except RuntimeError as err:
    print(str(err).split(";")[0])
else:
    raise AssertionError("ldlt should stop on the zero pivot")

x = mtl5.bunch_kaufman(swap).solve(mtl5.vector(np.array([2.0, 3.0]))).to_numpy()
print("bunch_kaufman: x =", x)
assert np.array_equal(x, [3.0, 2.0])

Which factorization, then:

| A is | Use | Python |
| --- | --- | --- |
| symmetric positive definite | Cholesky, the cheapest | `mtl5.cholesky(A)` |
| symmetric indefinite | LDLᵀ with pivoting | `mtl5.bunch_kaufman(A)`; `mtl5.ldlt(A)` doesn't pivot, and raises on a zero pivot |
| square, nothing more known | LU with partial pivoting | `mtl5.lu(A)`, or `mtl5.solve(A, b)` for one right-hand side |
| tall, a least-squares problem | QR | `mtl5.qr(A)` |

## Fitting a polynomial: QR against the normal equations

Fit a polynomial of degree p to 20 samples of sin x on [0, π], with the
tutorial's deterministic noise. The coefficients c minimize ‖Vc − y‖, where V
is the Vandermonde matrix with V[i, k] = xᵢᵏ.

There are two ways to solve it. The normal equations VᵀV c = Vᵀy are a small
SPD system, so Cholesky solves them. QR works on V itself. The difference is
the condition number: cond(VᵀV) = cond(V)², and the Vandermonde matrix's
condition number grows exponentially with the degree.

In [ ]:
m = 20
k = np.arange(m)
x_data = np.pi * k / (m - 1)
y_data = np.sin(x_data) + 0.02 * np.sin(7.0 * k + 3.0)


def vandermonde(p):
    return np.vander(x_data, p + 1, increasing=True)


def fit_qr(V):
    return mtl5.qr(V).solve(mtl5.vector(y_data)).to_numpy()


def fit_normal(V):
    # raises RuntimeError once rounding leaves V^T V not positive definite
    return mtl5.cholesky(V.T @ V).solve(mtl5.vector(V.T @ y_data)).to_numpy()


fig, ax = plt.subplots(figsize=(5.8, 3.6))
ax.plot(x_data, y_data, "o", color=INK_2, markersize=4, label="data")
fine = np.linspace(0.0, np.pi, 300)
for color, p in zip(SERIES, [1, 3, 8]):
    c = fit_qr(vandermonde(p))
    ax.plot(fine, np.polynomial.polynomial.polyval(fine, c), color=color, label=f"degree {p}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Least-squares polynomial fits to noisy sin x")
ax.legend(frameon=False)
ax.grid(True)
plt.tight_layout()
plt.show()

The plot can't show which method is more accurate: at these degrees the
fitted curves would lie on top of each other. The coefficients show it. To
measure their error, a reference solution is needed that is more accurate
than double precision. `mtl5.convert` puts V and y in quad-double, about 64
decimal digits, and the same `mtl5.qr` solves there. Rounded back to double,
it is the exact least-squares solution for this data, to double precision.

In [ ]:
degrees = np.arange(2, 14)
cond_V, err_qr, err_normal = [], [], []
for p in degrees:
    V = vandermonde(p)
    ref = mtl5.qr(mtl5.convert(V, "qd_cascade")).solve(mtl5.convert(y_data, "qd_cascade"))
    ref = np.asarray(ref.to_numpy(), dtype=float)

    def rel(c):
        return np.linalg.norm(c - ref) / np.linalg.norm(ref)

    cond_V.append(mtl5.condition_number(mtl5.matrix(V)))
    err_qr.append(rel(fit_qr(V)))
    try:
        err_normal.append(rel(fit_normal(V)))
    except RuntimeError:
        err_normal.append(np.nan)  # Cholesky failed: V^T V is not SPD in floating point
cond_V, err_qr, err_normal = map(np.array, (cond_V, err_qr, err_normal))

fig, ax = plt.subplots(figsize=(5.8, 3.8))
ax.semilogy(degrees, err_normal, "s-", color=SERIES[1], label="normal equations (Cholesky)")
ax.semilogy(degrees, err_qr, "o-", color=SERIES[0], label="QR")
ax.semilogy(degrees, cond_V**2 * eps, ":", color=SERIES[1], label="cond(V)² · ε")
ax.semilogy(degrees, cond_V * eps, ":", color=SERIES[0], label="cond(V) · ε")
ax.axhline(1.0, color=INK_2, linewidth=0.8)
ax.set_xlabel("polynomial degree p")
ax.set_ylabel("relative error of the coefficients")
ax.set_title("Error of the fitted coefficients, against quad-double")
ax.set_ylim(1e-16, 10.0)
ax.legend(frameon=False, fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

for p, kv, eq, en in zip(degrees, cond_V, err_qr, err_normal):
    print(f"p = {p:2d}  cond(V) = {kv:8.1e}  QR {eq:8.1e}  normal equations {en:8.1e}")

The rule of thumb holds: if cond(V) ≈ 10ᵏ, QR loses about k digits and the
normal equations about 2k. By degree 10 the normal equations have one or two
digits left, while QR still has nine. Past cond(V)² ≈ 1/ε, the normal
equations stop being definite in floating point, and Cholesky stops (the
missing points). The checks are on the bounds, not on the exact errors:

In [ ]:
# QR is backward stable: its error stays within a small multiple of cond(V) * eps
assert (err_qr <= 10 * cond_V * eps).all()

# from degree 6 on, the normal equations lose at least 100x more than QR,
# or fail outright
worse = degrees >= 6
assert (np.isnan(err_normal[worse]) | (err_normal[worse] >= 1e2 * err_qr[worse])).all()
print("QR within 10 cond(V) eps; normal equations at least 100x worse from degree 6")

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `lu_factor(A, pivot)`, `lu_solve(A, pivot, x, b)` | `F = mtl5.lu(A)`, `F.solve(b)` |
| `cholesky_factor(A)`, `cholesky_solve(A, x, b)` | `mtl5.cholesky(A).solve(b)` |
| `cholesky_factor(A)` returns the failing step | `mtl5.cholesky(A)` raises `RuntimeError` |
| `qr_factor(A, tau)`, `qr_solve(A, tau, x, b)` | `mtl5.qr(A).solve(b)`; `.Q` and `.R` are available |
| `inv(A) * b` | `mtl5.inv(A)`, best avoided |
| `trans(V) * V` | `V.T @ V` on the NumPy array |
| `eigenvalue_symmetric(VtV)`, then the ratio of extremes | `mtl5.condition_number(V)`, from the SVD |
| `dense2D<qd>` | `mtl5.convert(V, "qd_cascade")`, the same `mtl5.qr` |

The C++ factorizations overwrite A with the factors, so the program copies A
before each one. The Python ones leave A as it is.

Part 3 of [`phase05_dense_decompositions/solve_three_ways.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase05_dense_decompositions/solve_three_ways.cpp), ILU(0) and IC(0) as preconditioners, compares
iteration counts. It moves to the Krylov solvers notebook, once the Python
solvers report iterations (#147).